# Vectorized String & Regex Processing (5+ Years Interview Guide)
Exhaustive senior guide to .str accessor methods, regex extraction, string splitting expansions, pattern masking, and text cleaning.

### Key 5-Year Interview Concepts Covered:
- **Vectorized `.str` Accessor**: Element-wise string operations handling nulls (`NaN`) gracefully.
- **Regex Capture Groups (`str.extract`)**: Extracting structured numeric IDs and prefixes into dedicated DataFrame columns.
- **Column Splitting (`str.split(expand=True)`)**: Deconstructing delimited string fields into multi-column DataFrames.
- **String Replacement & Sanitization**: Removing currency symbols, whitespace padding, and irregular characters.
- **Vectorized String Joining (`str.cat`)**: Merging text columns with custom separators.

This interactive notebook is fully customized using the Fintech dataset `data/raw_transactions.csv`.

In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path, na_values=['Nan', ''])
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

## Section 1: Text Sanitization & Cleaning

### Vectorized Trimming & Case Normalization
**Explanation**: The `.str` accessor applies string transformations across entire Series in a single call. `.str.strip()` removes leading/trailing spaces, and `.str.upper()` / `.str.lower()` standardizes category casing. Unlike pure Python string operations, `.str` methods automatically preserve `NaN` values without raising `AttributeError`.

**Syntax**: `df['col'].str.strip().str.upper()`

In [ ]:
df['clean_region'] = df['region'].str.strip().str.capitalize()
print('Raw Regions:', df['region'].unique()[:5])
print('Cleaned Regions:', df['clean_region'].unique())

### Vectorized Pattern Matching (`str.contains`)
**Explanation**: `.str.contains(pat, regex=True, na=False)` tests whether a pattern or substring exists in each string, returning a boolean Series. Passing `na=False` treats `NaN` entries as `False` instead of propagating `NaN`, allowing direct use inside boolean filter masks.

**Syntax**: `df[df['card_type'].str.contains('Visa|Master', regex=True, na=False)]`

In [ ]:
visa_master_mask = df['card_type'].str.contains('Visa|MasterCard', regex=True, na=False)
print(df[visa_master_mask][['transaction_id', 'card_type']].head(4))

## Section 2: Regex Extraction & Token Splitting

### Regex Capture Groups with `str.extract()`
**Explanation**: `.str.extract(pat)` applies regular expressions containing capture groups `(...)` to extract structured fields into new DataFrame columns. For example, extracting numeric customer IDs `r'C(\d+)'` converts `'C82845'` into `'82845'`.

**Syntax**: `df['id'].str.extract(r'([A-Za-z]+)(\d+)')`

In [ ]:
extracted_ids = df['customer_id'].str.extract(r'([A-Za-z]+)(\d+)')
extracted_ids.columns = ['Prefix', 'Customer_Number']
print(extracted_ids.head(4))

### String Splitting into Multiple Columns (`str.split(expand=True)`)
**Explanation**: `.str.split(pat, expand=True)` splits strings on a delimiter and expands the results into separate DataFrame columns. This eliminates the need for manual loops when parsing comma-separated lists, key-value pairs, or compound headers.

**Syntax**: `df['col'].str.split('-', expand=True)`

In [ ]:
split_dates = df['transaction_date'].str.split('[-/ ]', expand=True)
print('Split Date Tokens:\n', split_dates.head(4))

### Vectorized String Concatenation (`str.cat`)
**Explanation**: `.str.cat(others, sep='_')` concatenates strings across multiple Series or a static separator element-wise in C. It handles missing values with the `na_rep` parameter to prevent null cascades.

**Syntax**: `df['col1'].str.cat(df['col2'], sep=' - ', na_rep='N/A')`

In [ ]:
df['tx_summary'] = df['transaction_id'].str.cat([df['card_type'], df['region']], sep=' | ', na_rep='Unknown')
print(df['tx_summary'].head(4))

## Section: Senior Fintech Interview Questions (5+ Years Experience)

### Q1: Extract Merchant Code & Digits with Regex
**Explanation**: Parse `merchant_id` (e.g. 'M2697') to separate the merchant category letter from the 4-digit terminal code using regex capture groups.

**Syntax**: `df['merchant_id'].str.extract(r'(?P<Category>[A-Z])(?P<Terminal>\d+)')`

In [ ]:
merchant_parsed = df['merchant_id'].str.extract(r'(?P<Category>[A-Z])(?P<Terminal>\d+)')
print(merchant_parsed.head(5))